## Task
Build an algorithm to assist ophthalmologists to grade retinal fundus images on a 5-level ordinal severity scale, classify diabetic retinopathy (DR). 

## Dataset
The dataset is in the folder `/data/` and is structured for PyTorch `ImageFolder` class. The images are organized in subfolders named `0`, `1`, `2`, `3`, and `4`, corresponding to the severity levels of diabetic retinopathy. Each subfolder contains the respective images for that severity level, either as `.jpg` or `.png` files.

## Approach
0. Adaptive Crop for Black Borders
    - Implement an adaptive cropping algorithm to remove black borders from the images, ensuring that the model focuses on the relevant retinal features.
1. Class Imbalances
   - Apply augmentations to minority classes until they have the same number of samples as the majority class.
   - Data augmentation techniques:
        - Horizontal/Vertical Flips
        - Random Rotations [-10, 10] degrees
        - Colour augmentation (random brightness, saturation, contrast modification from interval [−20, 20])
2. Preprocessing
   - For images 224x244, resize them to 384x384 pixels.
   - For images larger, resize to 512x512, then apply a center crop to 384x384 pixels.
   - Some image preprocessing before training
        - RGB-CLAHE: Apply Contrast Limited Adaptive Histogram Equalization (CLAHE) to enhance the contrast of the retinal images, which can help in highlighting important features for classification.
        - MaxGreenGsc-CLAHE: Apply CLAHE to the green channel of the images, as the green channel often contains more relevant information for retinal analysis.
        - Circular ROI Masking: Apply a circular mask to focus on the region of interest (ROI) in the retinal images, which can help in reducing noise and improving model performance.
        - Graham's: Apply Graham's method for image enhancement, which can help in improving the visibility of features in the retinal images.

3. Model Architecture

Model 1: ResNet-34
    - Use a pre-trained ResNet-34 architecture as the base model for feature extraction.
    - Modify the final fully connected layer to output 5 classes corresponding to the severity levels of diabetic retinopathy.

Model 2: EfficientNet-B4
    - Use a pre-trained EfficientNet-B4 architecture as the base model for feature extraction.
    - Modify the final fully connected layer to output 5 classes corresponding to the severity levels of diabetic retinopathy.

Model 3: Vision Transformer (ViT)
    - Use a pre-trained Vision Transformer (ViT) architecture as the base model for feature extraction.
    - Modify the final classification head to output 5 classes corresponding to the severity levels of diabetic retinopathy.

Model 4: Catboost
    - Train catboost model on the penultimate layer features of the above 3 models to further improve the classification performance.

Model 5: Ensemble Model
    - Combine the predictions from the 4 individual models (ResNet-34, EfficientNet-B4, and ViT, Catboost) using a weighted average or majority voting approach to improve overall classification performance.

4. Training
    - Train/val/test split: 70%/15%/15%
    - Use AdamW optimizer with a learning rate of 1e-4 and weight decay of 1e-5.
    - Implement a learning rate scheduler, OneCycleLR, to adjust the learning rate during training for better convergence.
    - Train the model for 20 epochs with a batch size of 16.
    - Use Cross-Entropy Loss as the loss function for multi-class classification.
    - Implement early stopping to prevent overfitting, monitoring the validation loss with a patience of 5 epochs.
    - Use mixed precision training to speed up the training process and reduce memory usage.
    - Gradient clipping with a max norm of 1.0 to prevent exploding gradients.
    - Implement model checkpointing to save the best model based on validation accuracy during training.
    - Print out metrics, F1-score after each epoch

---
## 1 · Imports & Environment

In [ ]:
import os, sys, random, warnings, itertools, gc, math
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, random_split
from torch.amp import autocast, GradScaler
import torchvision.transforms as T
from torchvision import models
from transformers import AutoModelForImageClassification

from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from IPython.display import FileLink

try:
    from catboost import CatBoostClassifier
    print('CatBoost available')
except ImportError:
    print('Installing catboost ...')
    os.system(f'{sys.executable} -m pip install catboost -q')
    from catboost import CatBoostClassifier
    print('CatBoost installed')

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('All imports OK')

## 2 · Configuration

In [ ]:
class CFG:
    seed         = 42
    device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    data_dir     = Path('')    # ImageFolder: clean_data/0 .. clean_data/4
    output_dir   = Path('')

    img_size     = 384
    batch_size   = 16
    num_workers  = 2
    epochs       = 20
    lr           = 2e-4
    weight_decay = 1e-5
    patience     = 5
    max_norm     = 0.5
    num_classes  = 5

    num_gpus     = torch.cuda.device_count()
    eff_batch    = batch_size * max(1, torch.cuda.device_count())

    # train / val / test split ratios
    train_ratio  = 0.90
    val_ratio    = 0.05
    test_ratio   = 0.05

CFG.output_dir.mkdir(exist_ok=True)

def seed_everything(seed: int = CFG.seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything()
print(f'Device : {CFG.device}  |  GPUs found: {CFG.num_gpus}')
print(f'Effective batch size: {CFG.eff_batch}')

## 3 · Dataset, Splits & DataLoaders

In [ ]:
class DRDataset(Dataset):
    def __init__(self, samples, transform=None, size=CFG.img_size):
        self.samples   = samples
        self.transform = transform
        self.size      = size

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.fromarray(np.zeros((self.size, self.size, 3), dtype=np.uint8))
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.long)


all_samples = []
for cls in sorted(os.listdir(CFG.data_dir)):
    cls_dir = CFG.data_dir / cls
    if not cls_dir.is_dir():
        continue
    lbl = int(cls)
    for fname in os.listdir(cls_dir):
        all_samples.append((str(cls_dir / fname), lbl))

print(f'Total images : {len(all_samples)}')

from sklearn.model_selection import train_test_split

paths, labels = zip(*all_samples)
paths, labels = list(paths), list(labels)

train_p, temp_p, train_l, temp_l = train_test_split(
    paths, labels, test_size=1 - CFG.train_ratio,
    stratify=labels, random_state=CFG.seed)

val_p, test_p, val_l, test_l = train_test_split(
    temp_p, temp_l, test_size=0.5,
    stratify=temp_l, random_state=CFG.seed)

train_samples = list(zip(train_p, train_l))
val_samples   = list(zip(val_p,   val_l))
test_samples  = list(zip(test_p,  test_l))

print(f'Train : {len(train_samples)}  |  Val : {len(val_samples)}  |  Test : {len(test_samples)}')

for name, lbls in [('train', train_l), ('val', val_l), ('test', test_l)]:
    counts = np.bincount(lbls, minlength=5)
    print(f'  {name:5s} -> {dict(enumerate(counts))}')

In [ ]:
train_transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    # T.RandomRotation(20),
    T.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.3, hue=0.2),
    # T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    # T.RandomErasing(p=0.2, scale=(0.02, 0.15)),
])

val_transform = T.Compose([
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = DRDataset(train_samples, transform=train_transform)
val_ds   = DRDataset(val_samples,   transform=val_transform)
test_ds  = DRDataset(test_samples,  transform=val_transform)

class_counts = np.bincount(train_l, minlength=5).astype(float)
class_weights = len(train_l) / (CFG.num_classes * class_counts)
class_weights[1] *= 3
class_weights[3] *= 2
sample_weights = [class_weights[l] for l in train_l]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights),
                                replacement=True)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(CFG.device)

print('Class weights:', {i: f'{w:.3f}' for i, w in enumerate(class_weights)})

train_loader = DataLoader(train_ds, batch_size=CFG.eff_batch,
                          sampler=sampler, num_workers=CFG.num_workers,
                          pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_ds, batch_size=CFG.eff_batch,
                          shuffle=False, num_workers=CFG.num_workers,
                          pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_ds, batch_size=CFG.eff_batch,
                          shuffle=False, num_workers=CFG.num_workers,
                          pin_memory=True, persistent_workers=True)

print(f'Batches  ->  train {len(train_loader)}  val {len(val_loader)}  test {len(test_loader)}')

## 4 · Model Definitions

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean', label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha          # class weight tensor
        self.reduction = reduction
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        num_classes = logits.size(1)
        
        # --- label smoothing ---
        with torch.no_grad():
            smooth_targets = torch.zeros_like(logits)
            smooth_targets.fill_(self.label_smoothing / (num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.label_smoothing)

        log_prob = F.log_softmax(logits, dim=1)            # (B, C)
        prob     = log_prob.exp()                           # (B, C)

        p_t      = prob.gather(1, targets.unsqueeze(1)).squeeze(1)   # (B,)
        focal_w  = (1.0 - p_t) ** self.gamma                        # (B,)

        ce = -(smooth_targets * log_prob).sum(dim=1)      # (B,)

        loss = focal_w * ce                               # (B,)

        if self.alpha is not None:
            alpha_t = self.alpha[targets]                 # (B,)
            loss = alpha_t * loss

        return loss.mean() if self.reduction == 'mean' else loss.sum()

In [ ]:
def build_resnet34(num_classes=CFG.num_classes):
    model = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(model.fc.in_features, num_classes),
    )
    return model


def build_efficientnet_b4(num_classes=CFG.num_classes):
    model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
    in_ftrs = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(in_ftrs, num_classes),
    )
    return model


def build_vit(num_classes=CFG.num_classes):
    base_model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_SWAG_E2E_V1)
    in_ftrs = base_model.heads.head.in_features
    base_model.heads.head = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_ftrs, num_classes)
    )
    
    class ViTWrapper(nn.Module):
        def __init__(self, vit):
            super().__init__()
            self.vit = vit
            self.heads = vit.heads
            
        def forward(self, x):
            return self.vit(x)
            
    return ViTWrapper(base_model)


class WideResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
            
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = torch.relu(out)
        return out


class DeepWideCNN(nn.Module):
    def __init__(self, num_classes=CFG.num_classes):
        super().__init__()
        self.in_channels = 128
        self.conv1 = nn.Conv2d(3, 128, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.layer1 = self._make_layer(128, blocks=2, stride=1)
        self.layer2 = self._make_layer(256, blocks=2, stride=2)
        self.layer3 = self._make_layer(512, blocks=3, stride=2)
        self.layer4 = self._make_layer(1024, blocks=3, stride=2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(1024, num_classes)
        )
        
    def _make_layer(self, out_channels, blocks, stride):
        layers = []
        layers.append(WideResBlock(self.in_channels, out_channels, stride))
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(WideResBlock(out_channels, out_channels))
        return nn.Sequential(*layers)
        
    def forward(self, x):
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x


def build_custom_cnn(num_classes=CFG.num_classes):
    return DeepWideCNN(num_classes=num_classes)

## 5. Training Utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler,
                    is_hf=False):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, leave=False, desc="train")
    for images, labels in pbar:
        images, labels = images.to(CFG.device), labels.to(CFG.device)

        optimizer.zero_grad()
        with autocast(device_type=CFG.device.type):
            if is_hf:
                outputs = model(images).logits
            else:
                outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CFG.max_norm)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        pbar.set_postfix(loss=f"{running_loss/total:.4f}",
                         acc=f"{correct/total:.4f}")

    return (running_loss / total,
            correct / total,
            f1_score(all_labels, all_preds, average="macro"))


@torch.no_grad()
def evaluate(model, loader, criterion, is_hf=False):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, leave=False, desc="eval"):
        images, labels = images.to(CFG.device), labels.to(CFG.device)
        with autocast(device_type=CFG.device.type):
            if is_hf:
                outputs = model(images).logits
            else:
                outputs = model(images)
            loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return (running_loss / total,
            correct / total,
            f1_score(all_labels, all_preds, average="macro"))


def train_model(model, name, is_hf=False, is_eval=False, val_f1_last=0, parallel=True):
    """Full training loop with early stopping & checkpointing."""

    checkpoint_path = CFG.output_dir / f"best_{name}.pth"
    if checkpoint_path.exists():
        print(f"Loading existing model from {checkpoint_path} ...")
        model.load_state_dict(
            torch.load(checkpoint_path, map_location=CFG.device, weights_only=True))
        print("Model loaded to continue")

        if is_eval:
            return model, None  # skip training, return loaded model
    
    model = model.to(CFG.device)

    if CFG.num_gpus > 1 and parallel is True:
        print(f"  [DataParallel] Wrapping '{name}' across {CFG.num_gpus} GPUs")
        model = nn.DataParallel(model)

    base = model.module if isinstance(model, nn.DataParallel) else model
    
    if hasattr(base, 'fc'):
        backbone_ratio = 0.1
        head_params    = list(base.fc.parameters())
    elif hasattr(base, 'classifier'):
        backbone_ratio = 0.5
        head_params    = list(base.classifier.parameters())
    elif hasattr(base, 'heads'):
        backbone_ratio = 0.2
        head_params    = list(base.vit.heads.parameters())
    else:
        backbone_ratio = 1.0
        head_params    = []

    head_ids        = {id(p) for p in head_params}
    backbone_params = [p for p in base.parameters() if id(p) not in head_ids]
    
    optimizer = optim.AdamW([
        {'params': backbone_params, 'lr': CFG.lr * backbone_ratio},
        {'params': head_params,     'lr': CFG.lr},
    ], weight_decay=CFG.weight_decay)
        
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    total_steps  = CFG.epochs * len(train_loader)
    warmup_steps = 2 * len(train_loader)  # 2-epoch warmup
    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return current_step / max(1, warmup_steps)   # linear warmup
        progress = (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))  # cosine, floor at 5%
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    
    scaler = GradScaler()

    best_val_f1 = val_f1_last
    patience_ctr = 0
    history = []

    for epoch in range(1, CFG.epochs + 1):
        tr_loss, tr_acc, tr_f1 = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, scaler,
            is_hf=is_hf)
        vl_loss, vl_acc, vl_f1 = evaluate(
            model, val_loader, criterion, is_hf=is_hf)

        history.append(dict(epoch=epoch,
                            tr_loss=tr_loss, tr_acc=tr_acc, tr_f1=tr_f1,
                            vl_loss=vl_loss, vl_acc=vl_acc, vl_f1=vl_f1))

        print(f"[{name}] Epoch {epoch:02d}/{CFG.epochs} | "
              f"Train  loss={tr_loss:.4f} acc={tr_acc:.4f} F1={tr_f1:.4f} | "
              f"Val  loss={vl_loss:.4f} acc={vl_acc:.4f} F1={vl_f1:.4f}")

        if vl_f1 > best_val_f1:
            best_val_f1  = vl_f1
            patience_ctr = 0
            state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(state, CFG.output_dir / f"best_{name}.pth")
            print(f"  -> saved best (val_f1={best_val_f1:.4f})")
        else:
            patience_ctr += 1

        if patience_ctr >= CFG.patience:
            print(f"  Early stopping at epoch {epoch}")
            break

    base = model.module if isinstance(model, nn.DataParallel) else model
    base.load_state_dict(
        torch.load(CFG.output_dir / f"best_{name}.pth",
                   map_location=CFG.device, weights_only=True))
    print(f"[{name}] Best val f1 = {best_val_f1:.4f}\n")

    return base, history

## 6. Training

In [ ]:
resnet34, hist_resnet = train_model(build_resnet34(), "resnet34", val_f1_last=0.6226) # batch size 64

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
FileLink(r'train_outputs/best_resnet34.pth')

In [ ]:
effnet_b4, hist_effnet = train_model(build_efficientnet_b4(), "efficientnet_b4", val_f1_last=0.5500, parallel=False) # batch size 32

In [ ]:
vit_model, hist_vit = train_model(build_vit(), "vit_torchvision", is_hf=False, parallel=False)

In [ ]:
cnn_model, cnn_hist = train_model(build_custom_cnn(), "custom_cnn")

## 7. Feature Extraction

In [ ]:
@torch.no_grad()
def extract_features(model, loader, model_type="torchvision"):
    model.eval()
    all_feats, all_labels = [], []

    features_buf = {}

    if model_type == "resnet":
        def hook_fn(m, inp, out):
            features_buf["feat"] = out.flatten(1)
        handle = model.avgpool.register_forward_hook(hook_fn)

    elif model_type == "efficientnet":
        def hook_fn(m, inp, out):
            features_buf["feat"] = out.flatten(1)
        handle = model.avgpool.register_forward_hook(hook_fn)

    elif model_type == "cnn":
        def hook_fn(m, inp, out):
            features_buf["feat"] = out.flatten(1)
        handle = model.avgpool.register_forward_hook(hook_fn)

    elif model_type == "vit":
        def hook_fn(m, inp):
            features_buf["feat"] = inp[0]
        handle = model.heads.register_forward_pre_hook(hook_fn)

    for images, labels in tqdm(loader, leave=False, desc=f"feat-{model_type}"):
        images = images.to(CFG.device)
        with autocast(device_type=CFG.device.type):
            _ = model(images)
        all_feats.append(features_buf["feat"].cpu().numpy())
        all_labels.append(labels.numpy())

    handle.remove()
    return np.concatenate(all_feats), np.concatenate(all_labels)


resnet34, _ = train_model(build_resnet34(), "resnet34", is_eval=True)
resnet34 = resnet34.to(CFG.device)

effnet_b4, _ = train_model(build_efficientnet_b4(), "efficientnet_b4", is_eval=True)
effnet_b4 = effnet_b4.to(CFG.device)

vit_model, _ = train_model(build_vit(), "vit_torchvision", is_hf=False, is_eval=True)
vit_model = vit_model.to(CFG.device)

print("Extracting features ...")
feat_train_r, lbl_train = extract_features(resnet34,  train_loader, "resnet")
feat_val_r,   lbl_val   = extract_features(resnet34,  val_loader,   "resnet")
feat_test_r,  lbl_test  = extract_features(resnet34,  test_loader,  "resnet")

feat_train_e, _ = extract_features(effnet_b4, train_loader, "efficientnet")
feat_val_e,   _ = extract_features(effnet_b4, val_loader,   "efficientnet")
feat_test_e,  _ = extract_features(effnet_b4, test_loader,  "efficientnet")

feat_train_v, _ = extract_features(vit_model, train_loader, "vit")
feat_val_v,   _ = extract_features(vit_model, val_loader,   "vit")
feat_test_v,  _ = extract_features(vit_model, test_loader,  "vit")

# feat_train_c, _ = extract_features(cnn_model, train_loader, "cnn")
# feat_val_c,   _ = extract_features(cnn_model, val_loader,   "cnn")
# feat_test_c,  _ = extract_features(cnn_model, test_loader,  "cnn")

## 8. Catboost

In [ ]:
X_train_cat = np.hstack([feat_train_r, feat_train_e, feat_train_v])
X_val_cat   = np.hstack([feat_val_r,   feat_val_e,   feat_val_v])
X_test_cat  = np.hstack([feat_test_r,  feat_test_e,  feat_test_v])

print(f"CatBoost input dims: train={X_train_cat.shape}  "
      f"val={X_val_cat.shape}  test={X_test_cat.shape}")

In [ ]:
cat_model = CatBoostClassifier(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    class_weights=dict(enumerate(class_weights)),
    eval_metric="Accuracy",
    random_seed=CFG.seed,
    verbose=100,
    early_stopping_rounds=50,
    task_type="GPU" if torch.cuda.is_available() else "CPU",
)

cat_model.fit(
    X_train_cat, lbl_train,
    eval_set=(X_val_cat, lbl_val),
    use_best_model=True,
)

cat_val_pred  = cat_model.predict(X_val_cat).flatten().astype(int)
cat_test_pred = cat_model.predict(X_test_cat).flatten().astype(int)

print(f"\n[CatBoost] Val  acc={accuracy_score(lbl_val, cat_val_pred):.4f}  "
      f"F1={f1_score(lbl_val, cat_val_pred, average='macro'):.4f}")
print(f"[CatBoost] Test acc={accuracy_score(lbl_test, cat_test_pred):.4f}  "
      f"F1={f1_score(lbl_test, cat_test_pred, average='macro'):.4f}")

cat_model.save_model(str(CFG.output_dir / "best_catboost.cbm"))
print("CatBoost model saved")

### Catboost without ViT

In [ ]:
X_train_cat_1 = np.hstack([feat_train_r, feat_train_e ])
X_val_cat_1   = np.hstack([feat_val_r,   feat_val_e])
X_test_cat_1  = np.hstack([feat_test_r,  feat_test_e])

In [ ]:
cat_model = CatBoostClassifier(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    class_weights=dict(enumerate(class_weights)),
    eval_metric="Accuracy",
    random_seed=CFG.seed,
    verbose=100,
    early_stopping_rounds=100,
    task_type="GPU" if torch.cuda.is_available() else "CPU",
)

cat_model.fit(
    X_train_cat_1, lbl_train,
    eval_set=(X_val_cat_1, lbl_val),
    use_best_model=True,
)

cat_val_pred  = cat_model.predict(X_val_cat_1).flatten().astype(int)
cat_test_pred = cat_model.predict(X_test_cat_1).flatten().astype(int)

print(f"\n[CatBoost] Val  acc={accuracy_score(lbl_val, cat_val_pred):.4f}  "
      f"F1={f1_score(lbl_val, cat_val_pred, average='macro'):.4f}")
print(f"[CatBoost] Test acc={accuracy_score(lbl_test, cat_test_pred):.4f}  "
      f"F1={f1_score(lbl_test, cat_test_pred, average='macro'):.4f}")

cat_model.save_model(str(CFG.output_dir / "best_catboost_novit.cbm"))
print("CatBoost model saved")

In [ ]:
evaluate_and_plot(test_labels, cat_test_pred, "CatBoost")

## 9. Eval

In [ ]:
@torch.no_grad()
def get_test_predictions(model, loader, is_hf=False):
    model.eval()
    all_preds = []
    all_labels = []
    for images, labels in tqdm(loader, leave=False, desc="Predicting"):
        images = images.to(CFG.device)
        with autocast(device_type=CFG.device.type):
            logits = model(images).logits if is_hf else model(images)
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_preds), np.array(all_labels)

def evaluate_and_plot(y_true, y_pred, model_name):
    print(f"\n{'='*50}\n{model_name} Evaluation\n{'='*50}")
    print(classification_report(y_true, y_pred, digits=4))
    f1 = f1_score(y_true, y_pred, average="macro")
    print(f"Macro F1 Score: {f1:.4f}")
    
    # Normalized confusion matrix
    cm = confusion_matrix(y_true, y_pred, normalize='true')
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=range(CFG.num_classes), 
                yticklabels=range(CFG.num_classes), 
                vmin=0, vmax=1)
    plt.title(f"{model_name} - Normalized Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label (Normalized)")
    plt.tight_layout()
    plt.show()

print("Getting predictions for deep learning models on Test Set...")
resnet34, _ = train_model(build_resnet34(), "resnet34", is_eval=True)
resnet34 = resnet34.to(CFG.device)

effnet_b4, _ = train_model(build_efficientnet_b4(), "efficientnet_b4", is_eval=True)
effnet_b4 = effnet_b4.to(CFG.device)

vit_model, _ = train_model(build_vit(), "vit_torchvision", is_hf=False, is_eval=True)
vit_model = vit_model.to(CFG.device)

# cnn_model, _ = train_model(build_custom_cnn(), "custom_cnn", is_eval=True)
# cnn_model = cnn_model.to(CFG.device)

cat_model = CatBoostClassifier(task_type="GPU" if torch.cuda.is_available() else "CPU")
cat_model.load_model(str(CFG.output_dir / "best_catboost.cbm"))

resnet_test_preds, test_labels = get_test_predictions(resnet34, test_loader)
effnet_test_preds, _ = get_test_predictions(effnet_b4, test_loader)
vit_test_preds, _ = get_test_predictions(vit_model, test_loader, is_hf=False)
# cnn_test_preds, _ = get_test_predictions(cnn_model, test_loader)

# Plot for each model one by one
evaluate_and_plot(test_labels, resnet_test_preds, "ResNet-34")
evaluate_and_plot(test_labels, effnet_test_preds, "EfficientNet-B4")
evaluate_and_plot(test_labels, vit_test_preds, "Vision Transformer (ViT)")
# evaluate_and_plot(test_labels, cnn_test_preds, "Custom CNN")
evaluate_and_plot(test_labels, cat_test_pred, "CatBoost")

## 10. Ensemble

In [ ]:
@torch.no_grad()
def get_probas(model, loader, is_hf=False):
    """Get softmax probabilities from a DL model."""
    model.eval()
    all_probs = []
    for images, _ in tqdm(loader, leave=False):
        images = images.to(CFG.device)
        with autocast(device_type=CFG.device.type):
            logits = model(images).logits if is_hf else model(images)
        all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(all_probs)

print("Collecting validation probabilities to train meta-model ...")
val_prob_resnet = get_probas(resnet34,  val_loader)
val_prob_effnet = get_probas(effnet_b4, val_loader)
val_prob_vit    = get_probas(vit_model, val_loader, is_hf=False)
# val_prob_cnn    = get_probas(cnn_model, val_loader, is_hf=False)
val_prob_cat    = cat_model.predict_proba(X_val_cat)

X_val_meta = np.hstack([val_prob_resnet, val_prob_effnet, val_prob_vit, val_prob_cat])

# Train a Meta-Model
print("Training Logistic Regression meta-model (Stacking) ...")
meta_model = LogisticRegression(max_iter=1000, random_state=CFG.seed)
meta_model.fit(X_val_meta, lbl_val)

print("Collecting test probabilities ...")
prob_resnet = get_probas(resnet34,  test_loader)
prob_effnet = get_probas(effnet_b4, test_loader)
prob_vit    = get_probas(vit_model, test_loader, is_hf=False)
# prob_cnn    = get_probas(cnn_model, test_loader, is_hf=False)
prob_cat    = cat_model.predict_proba(X_test_cat)

X_test_meta = np.hstack([prob_resnet, prob_effnet, prob_vit, prob_cat])

ensemble_probs = meta_model.predict_proba(X_test_meta)
ensemble_preds = meta_model.predict(X_test_meta)

models_results = {}
for mname, probs in [("ResNet-34",       prob_resnet),
                     ("EfficientNet-B4", prob_effnet),
                     ("ViT",             prob_vit),
                     # ("Custom CNN",      prob_cnn),
                     ("CatBoost",        prob_cat),
                     ("Ensemble",        ensemble_probs)]:
    preds = probs.argmax(axis=1)
    acc = accuracy_score(lbl_test, preds)
    f1  = f1_score(lbl_test, preds, average="macro")
    models_results[mname] = {"Accuracy": acc, "Macro-F1": f1}
    print(f"  {mname:20s}  acc={acc:.4f}  F1={f1:.4f}")

print("DETAILED ENSEMBLE CLASSIFICATION REPORT")
print(classification_report(lbl_test, ensemble_preds, digits=4))

In [ ]:
import joblib
lr_save_path = CFG.output_dir / "best_lr.pkl"
joblib.dump(meta_model, lr_save_path)

In [ ]:
cm = confusion_matrix(lbl_test, ensemble_preds, normalize='true')
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=range(5), yticklabels=range(5))
plt.title("Ensemble - Test Confusion Matrix")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout(); plt.show()

pd.DataFrame(models_results).T.to_csv(CFG.output_dir / "final_results.csv")
print(f"\nResults saved to {CFG.output_dir / 'final_results.csv'}")

## 11. Ensemble with Rf, Lr, SoftVoting

In [ ]:
@torch.no_grad()
def get_probas(model, loader, is_hf=False):
    """Get softmax probabilities from a DL model."""
    model.eval()
    all_probs = []
    for images, _ in tqdm(loader, leave=False):
        images = images.to(CFG.device)
        with autocast(device_type=CFG.device.type):
            logits = model(images).logits if is_hf else model(images)
        all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(all_probs)

print("Collecting validation probabilities to train meta-models ...")
val_prob_resnet = get_probas(resnet34,  val_loader)
val_prob_effnet = get_probas(effnet_b4, val_loader)
val_prob_vit    = get_probas(vit_model, val_loader, is_hf=False)
val_prob_cnn    = get_probas(cnn_model, val_loader, is_hf=False)
val_prob_cat    = cat_model.predict_proba(X_val_cat)

print("Collecting test probabilities ...")
prob_resnet = get_probas(resnet34,  test_loader)
prob_effnet = get_probas(effnet_b4, test_loader)
prob_vit    = get_probas(vit_model, test_loader, is_hf=False)
prob_cnn    = get_probas(cnn_model, test_loader, is_hf=False)
prob_cat    = cat_model.predict_proba(X_test_cat)

In [ ]:
val_prob_cat = cat_model.predict_proba(X_val_cat)
prob_cat    = cat_model.predict_proba(X_test_cat)

In [ ]:
val_probs = {
    "ResNet-34": val_prob_resnet,
    "EfficientNet-B4": val_prob_effnet,
    "ViT": val_prob_vit,
    # "Custom CNN": val_prob_cnn,
    "CatBoost": val_prob_cat
}

test_probs = {
    "ResNet-34": prob_resnet,
    "EfficientNet-B4": prob_effnet,
    "ViT": prob_vit,
    # "Custom CNN": prob_cnn,
    "CatBoost": prob_cat
}

model_names = list(val_probs.keys())
best_f1 = 0
best_combo = None
best_method = None
best_test_preds = None

results = []

print("Evaluating all combinations of models and ensemble methods...")

# Iterate through combinations of 2 to 5 models
for r in range(2, len(model_names) + 1):
    for combo in itertools.combinations(model_names, r):
        combo_names = list(combo)
        combo_str = " + ".join(combo_names)
        
        X_val_combo = np.hstack([val_probs[name] for name in combo_names])
        X_test_combo = np.hstack([test_probs[name] for name in combo_names])
        
        val_mean_prob = np.mean([val_probs[name] for name in combo_names], axis=0)
        test_mean_prob = np.mean([test_probs[name] for name in combo_names], axis=0)
        
        test_preds_mean = test_mean_prob.argmax(axis=1)
        acc_mean = accuracy_score(lbl_test, test_preds_mean)
        f1_mean = f1_score(lbl_test, test_preds_mean, average="macro")
        
        results.append({
            "Combination": combo_str,
            "Method": "Soft Voting (Mean)",
            "Test Accuracy": acc_mean,
            "Test Macro-F1": f1_mean
        })
        
        if f1_mean > best_f1:
            best_f1 = f1_mean
            best_combo = combo_names
            best_method = "Soft Voting (Mean)"
            best_test_preds = test_preds_mean
            
        lr_meta = LogisticRegression(max_iter=1000, random_state=CFG.seed)
        lr_meta.fit(X_val_combo, lbl_val)
        test_preds_lr = lr_meta.predict(X_test_combo)
        
        acc_lr = accuracy_score(lbl_test, test_preds_lr)
        f1_lr = f1_score(lbl_test, test_preds_lr, average="macro")
        
        results.append({
            "Combination": combo_str,
            "Method": "Logistic Regression Stacking",
            "Test Accuracy": acc_lr,
            "Test Macro-F1": f1_lr
        })
        
        if f1_lr > best_f1:
            best_f1 = f1_lr
            best_combo = combo_names
            best_method = "Logistic Regression Stacking"
            best_test_preds = test_preds_lr
            
        rf_meta = RandomForestClassifier(n_estimators=100, random_state=CFG.seed, n_jobs=-1)
        rf_meta.fit(X_val_combo, lbl_val)
        test_preds_rf = rf_meta.predict(X_test_combo)
        
        acc_rf = accuracy_score(lbl_test, test_preds_rf)
        f1_rf = f1_score(lbl_test, test_preds_rf, average="macro")
        
        results.append({
            "Combination": combo_str,
            "Method": "Random Forest Stacking",
            "Test Accuracy": acc_rf,
            "Test Macro-F1": f1_rf
        })
        
        if f1_rf > best_f1:
            best_f1 = f1_rf
            best_combo = combo_names
            best_method = "Random Forest Stacking"
            best_test_preds = test_preds_rf

# Convert results to DataFrame and display top 15
results_df = pd.DataFrame(results).sort_values(by="Test Macro-F1", ascending=False)
print("\nTop 15 Model Combinations & Methods:")
print(results_df.head(15).to_string(index=False))

In [ ]:
print(f"Best Combination: {' + '.join(best_combo)}")
print(f"Best Method: {best_method}")
print(f"Best Test Macro-F1: {best_f1:.4f}")
print("\nSingle Model Baselines:")
models_results = {}

for mname, probs in test_probs.items():
    preds = probs.argmax(axis=1)
    acc = accuracy_score(lbl_test, preds)
    f1  = f1_score(lbl_test, preds, average="macro")
    models_results[mname] = {"Accuracy": acc, "Macro-F1": f1}
    print(f"  {mname:20s}  acc={acc:.4f}  F1={f1:.4f}")

print(f"DETAILED REPORT: BEST ENSEMBLE")
print(f"Models: {' + '.join(best_combo)} | Method: {best_method}")
print(classification_report(lbl_test, best_test_preds, digits=4))

# Confusion matrix for the best ensemble
cm = confusion_matrix(lbl_test, best_test_preds, normalize='true')
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=range(5), yticklabels=range(5))
plt.title(f"Best Ensemble - Test Confusion Matrix\n({' + '.join(best_combo)} | {best_method})")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout(); plt.show()

results_df.to_csv(CFG.output_dir / "ensemble_combinations_results.csv", index=False)
print(f"\nFull ensemble results saved to {CFG.output_dir / 'ensemble_combinations_results.csv'}")